In [ ]:
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Machine Learning

1. Risk Stratification (Classification)
Early disease detection using tabular health indicators (age, BMI, labs, history)
2. Length of Stay Prediction (Regression)
Forecasting hospitalization duration from vitals, diagnosis codes, procedures
3. Patient Segmentation (Clustering)
Grouping patients into cohorts based on similarity in health and behavior
4. Medical Associations (Association Rules)
Mining co-occurrence patterns like BMI + hypertension → diabetes risk

1. https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction?resource=download
2. https://www.kaggle.com/datasets/aayushchou/hospital-length-of-stay-dataset-microsoft

## Risk Stratification

In [ ]:
import pandas as pd

risk_stratification_df = pd.read_csv('/content/risk_stratification.csv')

In [ ]:
risk_stratification_df.head()

# EDA <PLACE HOLDER>

In [ ]:
risk_stratification_df.shape

In [ ]:
risk_stratification_df.HeartDisease.value_counts()

In [ ]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
y = risk_stratification_df["HeartDisease"]
X = risk_stratification_df.drop("HeartDisease", axis=1)

In [ ]:
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

In [ ]:
num_features

In [ ]:
cat_features

In [ ]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

In [ ]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
model_log = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
model_rf = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200,
                                          random_state=42))
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model_log.fit(X_train, y_train)

In [ ]:
model_rf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
y_pred = model_log.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

plt.figure()
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
y_pred = model_rf.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

plt.figure()
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
new_patient = pd.DataFrame([{
    "Age": 45,
    "Sex": "M",
    "ChestPainType": "ATA",
    "RestingBP": 135,
    "Cholesterol": 250,
    "FastingBS": 0,
    "RestingECG": "Normal",
    "MaxHR": 160,
    "ExerciseAngina": "N",
    "Oldpeak": 0.5,
    "ST_Slope": "Up"
}])

In [ ]:
risk = model_log.predict(new_patient)
prob = model_log.predict_proba(new_patient)

print("Risk:", "High" if risk[0]==1 else "Low")
print("Probability:", prob)

In [ ]:
risk = model_rf.predict(new_patient)
prob = model_rf.predict_proba(new_patient)

print("Risk:", "High" if risk[0]==1 else "Low")
print("Probability:", prob)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

In [ ]:
y_pred = model_rf.predict(X_test)
y_prob = model_rf.predict_proba(X_test)[:, 1]   # Probability of class = 1

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", accuracy)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

In [ ]:
import joblib
joblib.dump(model_log, 'q1_model_lr.pkl')
print("✅ LOS model saved!")

joblib.dump(model_rf, 'q1_model_rf.pkl')
print("✅ LOS model saved!")


## Length of Stay

In [ ]:
duration_forecasting_df = pd.read_csv('/content/duration_forecasting.csv')


duration_forecasting_df = duration_forecasting_df.sample(
    n=10000,
    random_state=42
)

In [ ]:
duration_forecasting_df.head()

In [ ]:
y = duration_forecasting_df["lengthofstay"]
X = duration_forecasting_df.drop(["lengthofstay",'vdate','discharged'], axis=1)

In [ ]:
X.columns

In [ ]:
X["rcount"] = X["rcount"].replace("5+", 5).astype(int)

In [ ]:
num_features = X.select_dtypes(include=["int64","float64"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [ ]:
from sklearn.linear_model import LinearRegression

model_los_lr = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", LinearRegression())
])

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_los_rf = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor())
])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model_los_lr.fit(X_train, y_train)

In [ ]:
model_los_rf.fit(X_train, y_train)

In [ ]:
y_pred = model_los_lr.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
plt.figure()

plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle="--")

plt.xlabel("Actual Length of Stay")
plt.ylabel("Predicted Length of Stay")
plt.title("Actual vs Predicted LOS")

plt.show()

In [ ]:
y_pred = model_los_rf.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
plt.figure()

plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle="--")

plt.xlabel("Actual Length of Stay")
plt.ylabel("Predicted Length of Stay")
plt.title("Actual vs Predicted LOS")

plt.show()

## Patient Segmentation

In [ ]:
np.random.seed(42)

# Load data
duration_forecasting_df = pd.read_csv('/content/duration_forecasting.csv')

duration_forecasting_df = duration_forecasting_df.sample(
    n=10000,
    random_state=42
)

id_cols = ["eid", "facid","vdate","discharged"]   # IDs
target_cols = ["lengthofstay"]  # outcome

df_cluster = duration_forecasting_df.drop(id_cols + target_cols, axis=1)

In [ ]:
id_cols = ["eid", "facid","vdate","discharged"]
target_cols = ["lengthofstay"]

df_cluster = duration_forecasting_df.drop(
    columns=id_cols + target_cols,
    errors="ignore"
)

In [ ]:
if "rcount" in df_cluster.columns:
    df_cluster["rcount"] = (
        df_cluster["rcount"]
        .replace("5+", 5)
        .astype(float)
    )

In [ ]:
num_cols = df_cluster.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols = df_cluster.select_dtypes(
    include=["object"]
).columns

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

In [ ]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [ ]:
X_scaled = preprocessor.fit_transform(df_cluster)

print("Processed Shape:", X_scaled.shape)

In [ ]:
from sklearn.cluster import KMeans

wcss = []

k_range = range(2, 40)

for k in k_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    wcss.append(kmeans.inertia_)

In [ ]:
plt.figure()

plt.plot(k_range, wcss, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method for Optimal K")

plt.show()

In [ ]:
best_k = 20

In [ ]:
final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = final_kmeans.fit_predict(X_scaled)

In [ ]:
df_clustered = df_cluster.copy()

df_clustered["cluster"] = cluster_labels

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score

sil_score = silhouette_score(X_scaled, cluster_labels)

ch_score = calinski_harabasz_score(X_scaled, cluster_labels)

print("Silhouette Score:", sil_score)
print("Calinski-Harabasz Score:", ch_score)

In [ ]:
print("\nCluster Counts:")
print(df_clustered["cluster"].value_counts())

In [ ]:
num_cols2 = df_clustered.select_dtypes(
    include=["int64", "float64"]
).columns

cluster_summary = (
    df_clustered
    .groupby("cluster")[num_cols2]
    .mean()
)

cluster_summary

## Medical Association

In [ ]:
!pip install mlxtend

In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
df = pd.read_csv('/content/duration_forecasting.csv')

#df = df.sample(n=10000, random_state=42)

In [ ]:
comorb_cols = [
    "asthma",
    "dialysisrenalendstage",
    "irondef",
    "pneum",
    "substancedependence",
    "psychologicaldisordermajor"
]

vital_cols = [
    "bmi",
    "glucose",
    "creatinine",
    "pulse"
]


df_assoc = df[comorb_cols + vital_cols].copy()

In [ ]:
imputer = SimpleImputer(strategy="median")

df_assoc[vital_cols] = imputer.fit_transform(
    df_assoc[vital_cols]
)

In [ ]:
df_assoc[comorb_cols] = df_assoc[comorb_cols].fillna(0)

In [ ]:
scaler = StandardScaler()

df_assoc[vital_cols] = scaler.fit_transform(
    df_assoc[vital_cols]
)

In [ ]:
df_assoc["high_bmi"] = (df_assoc["bmi"] > 1).astype(int)
df_assoc["high_glucose"] = (df_assoc["glucose"] > 1).astype(int)
df_assoc["high_pulse"] = (df_assoc["pulse"] > 1).astype(int)
df_assoc["high_creatinine"] = (df_assoc["creatinine"] > 1).astype(int)

In [ ]:
binary_cols = (
    comorb_cols +
    ["high_bmi","high_glucose","high_pulse","high_creatinine"]
)

transactions = df_assoc[binary_cols]

In [ ]:
transactions

In [ ]:
frequent_itemsets = apriori(
    transactions,
    min_support=0.01,   # appears in 5% patients
    use_colnames=True
)

frequent_itemsets

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

In [ ]:
rules

In [ ]:
rules = rules.sort_values(
    by="lift",
    ascending=False
)
rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(10)

# Deep Learning

1. Imaging Diagnostics (CNN)
X-ray, CT, MRI analysis for disease detection and staging
2. Sequence Modeling (RNN / LSTM)
Time-series modeling of vitals, labs, ICU signals for deterioration prediction
3. Sentiment Analysis (deep-learning version)
Neural text models (LSTM/CNN/Transformers) applied to patient feedback

## Imaging Diagnostics

1. https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
2. https://www.kaggle.com/datasets/shayanfazeli/heartbeat
3. https://www.kaggle.com/datasets/junaid6731/hospital-reviews-dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

In [ ]:
pip install tensorflow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [ ]:
BASE_PATH = "/kaggle/input/chest-xray-pneumonia/chest_xray"

TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")
VAL_PATH   = os.path.join(BASE_PATH, "val")

In [ ]:
BASE_PATH = "/kaggle/input/chest-xray-pneumonia/chest_xray"

TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")
VAL_PATH   = os.path.join(BASE_PATH, "val")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

In [ ]:
val_data = val_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

In [ ]:
test_data = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False   # Freeze layers

In [ ]:
model_s = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

In [ ]:
model_s.compile(
    optimizer="adam",
    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall(),
        tf.keras.metrics.AUC()
    ]
)

In [ ]:
model_s.summary()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
EPOCHS = 3

history = model_s.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

In [ ]:
# Already there: cnn_model.save('pneumoniamodel.keras')
# Rename if needed:
model_s.save('pneumonia_cnn_final.keras')  # After history plot (cell ~101)
print("✅ CNN saved as pneumonia_cnn_final.keras")


In [ ]:
results = model_s.evaluate(test_data)

print("\nTest Results:")
print("Loss:", results[0])
print("Accuracy:", results[1])
print("Precision:", results[2])
print("Recall:", results[3])
print("AUC:", results[4])

In [ ]:
# Accuracy
plt.figure()
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])
plt.show()


# Loss
plt.figure()
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train", "Validation"])
plt.show()

In [ ]:
from tensorflow.keras.preprocessing import image

In [ ]:
sample_img = TEST_PATH + "/PNEUMONIA/person1_virus_6.jpeg"

In [ ]:
img = image.load_img(
    sample_img,
    target_size=IMG_SIZE
)

img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model_s.predict(img_array)[0][0]

print("Pneumonia Probability:", pred)

if pred > 0.5:
    print("Prediction: PNEUMONIA")
else:
    print("Prediction: NORMAL")

## Sequence Modelling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shayanfazeli/heartbeat")

print("Path to dataset files:", path)

In [ ]:
import os

DATA_PATH = "/kaggle/input/heartbeat"

print(os.listdir(DATA_PATH))

In [ ]:
train_df = pd.read_csv(
    DATA_PATH + "/mitbih_train.csv",
    header=None
)

test_df = pd.read_csv(
    DATA_PATH + "/mitbih_test.csv",
    header=None
)

In [ ]:
df = pd.concat([train_df, test_df], axis=0)

# Shuffle
df = df.sample(frac=1, random_state=42)

# Take only 10K rows (small version)
df = df.sample(n=10000, random_state=42)

print("Final Shape:", df.shape)

In [ ]:
X = df.iloc[:, :-1].values   # ECG signal (187 points)
y = df.iloc[:, -1].values   # Class

In [ ]:
y = (y != 0).astype(int)

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
X_lstm = X_scaled.reshape(
    X_scaled.shape[0],
    X_scaled.shape[1],
    1
)

print("LSTM Shape:", X_lstm.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model_seq = Sequential([

    LSTM(
        64,
        return_sequences=True,
        input_shape=(187, 1)
    ),

    Dropout(0.3),

    LSTM(32),

    Dense(1, activation="sigmoid")
])

In [ ]:
model_seq.compile(
    optimizer="adam",
    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall(),
        tf.keras.metrics.AUC()
    ]
)

In [ ]:
history = model_seq.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test)
)

In [ ]:
results = model_seq.evaluate(X_test, y_test)

print("\nTest Results")
print("Loss:", results[0])
print("Accuracy:", results[1])
print("Precision:", results[2])
print("Recall:", results[3])
print("AUC:", results[4])

In [ ]:
y_pred = (model_seq.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

In [ ]:
# Accuracy
plt.figure()
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])
plt.show()


# Loss
plt.figure()
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train", "Validation"])
plt.show()

In [ ]:
model_seq.save("model_lstm.keras")

## Sentiment Analysis

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("junaid6731/hospital-reviews-dataset")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import precision_score, recall_score, matthews_corrcoef

In [ ]:
import os

DATA_PATH = "/kaggle/input/hospital-reviews-dataset"

print(os.listdir(DATA_PATH))

In [ ]:
df = pd.read_csv(DATA_PATH + "/hospital.csv")
print(df.head())
print(df.shape)

In [ ]:
df = df.drop(columns=["Unnamed: 3"], errors="ignore")

In [ ]:
df = df.rename(columns={
    "Feedback": "text",
    "Sentiment Label": "label"
})

In [ ]:
df = df.dropna(subset=["text", "label"])

In [ ]:
print(df.head())
print(df["label"].value_counts())

In [ ]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
dataset = Dataset.from_pandas(df)

In [ ]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=128
    )


dataset = dataset.map(tokenize, batched=True)

In [ ]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

In [ ]:
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
training_args = TrainingArguments(

    output_dir="./bert_patient_reviews",

    eval_strategy="epoch",

    per_device_train_batch_size=8,   # small dataset
    per_device_eval_batch_size=8,

    num_train_epochs=5,   # more epochs (small data)

    learning_rate=2e-5,

    logging_steps=20,

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [ ]:
trainer.train()

In [ ]:
# Already: model.save_pretrained('sentimentmodeldir')
# Zip it:
!zip -r sentiment_model.zip sentimentmodeldir/
print("✅ BERT zipped!")


In [ ]:
preds = trainer.predict(dataset["test"])

y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

In [ ]:
from sklearn.metrics import precision_score, recall_score, matthews_corrcoef, accuracy_score

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("MCC      :", mcc)

In [ ]:
def predict_sentiment(text):

    device = next(model.parameters()).device   # detect GPU/CPU

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()   # move back to CPU for printing

In [ ]:
review = "Doctors were careless and nurses were rude"

print(predict_sentiment(review))

In [ ]:
def predict_sentiment_label(text):

    probs = predict_sentiment(text)[0]

    if probs[1] > probs[0]:
        return "Positive Review", probs
    else:
        return "Negative Review", probs


print(predict_sentiment_label(review))

In [ ]:
save_path = "./clinicalbert_model"

tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

**Pickling**